# Percobaan 01: Preprocessing Citra
Notebook ini berisi tahap awal pengolahan citra sebelum data digunakan untuk ekstraksi fitur dan klasifikasi. Proses yang dilakukan meliputi pembacaan dataset, penyeragaman ukuran citra, konversi grayscale, filtering, peningkatan kontras, deteksi tepi, thresholding, dan penyimpanan hasil. Setiap metode preprocessing dibuat sebagai fungsi terpisah agar hasilnya dapat dibandingkan pada tahap percobaan berikutnya.


## Import Library
Bagian ini mengimpor library yang dibutuhkan untuk menjalankan proses preprocessing citra. Library `os` dan `pathlib` digunakan untuk mengakses folder dataset serta mengatur lokasi penyimpanan hasil. Library `cv2` digunakan untuk membaca, mengubah ukuran, mengonversi warna, dan menyimpan citra. Library `numpy` digunakan untuk operasi matriks, pembuatan kernel, dan pengolahan nilai piksel. Library `matplotlib` disiapkan untuk kebutuhan visualisasi apabila hasil citra perlu ditampilkan.


In [1]:
import os
import cv2 as cv
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

## Pendefinisian Kernel
Bagian ini mendefinisikan kernel yang digunakan pada proses filtering dan deteksi tepi. `kernelSharpening` dibuat untuk menonjolkan detail citra, sedangkan `sobelX` dan `sobelY` digunakan untuk menghitung perubahan intensitas pada arah horizontal dan vertikal. Kedua kernel Sobel tersebut nantinya dipakai bersama pada fungsi deteksi tepi. `kernel_diamond` berbentuk pola berlian dan disiapkan untuk operasi morfologi seperti dilasi atau thickening. Semua kernel dibuat dalam bentuk array agar dapat langsung digunakan dalam operasi berbasis matriks.


In [7]:
kernelSharpening = np.array([
    [1/9, 1/9, 1/9],
    [1/9, 8/9, 1/9],
    [1/9, 1/9, 1/9]
], dtype=np.float32)

sobelX = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

sobelY = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

kernel_diamond = np.array([
    [0,0,1,0,0],
    [0,1,1,1,0],
    [1,1,1,1,1],
    [0,1,1,1,0],
    [0,0,1,0,0]
])

## Fungsi Dasar Preprocessing
Bagian ini berisi beberapa fungsi dasar yang digunakan dalam proses preprocessing citra. Fungsi `thresholding()` mengubah citra grayscale menjadi citra biner berdasarkan nilai ambang tertentu. Fungsi `filter()` menyediakan mean filter, median filter, dan modus filter untuk menghaluskan citra atau mengurangi noise. Fungsi `convolution()` digunakan untuk menerapkan kernel pada citra, sedangkan `edge_detection()` menggabungkan respons Sobel arah X dan Y untuk menghasilkan citra tepi. Fungsi `dilasi()` dan `thickening()` digunakan untuk mempertebal area putih pada citra biner, sementara `ekualisasi()` meningkatkan kontras citra melalui perhitungan histogram dan CDF.


In [8]:
def thresholding(img, batas):
    baris, kolom = img.shape
    canvas = np.zeros_like(img, dtype=np.uint8)

    for i in range(baris):
        for j in range(kolom):
            if(img[i,j] > batas):
                canvas[i,j] = 255
            elif(img[i,j] <= batas):
                canvas[i,j] = 0

    return canvas

def filter(img, size, mode):
    # dimensi gambar
    height, width = img.shape

    # ukuran padding
    pad = size // 2

    # tambah padding di sisi gambar
    padded = np.pad(img, pad, mode='edge')

    # canvas hasil
    canvas = np.zeros_like(img, dtype=np.uint8)

    match mode:

        # FILTER RATA-RATA / MEAN FILTER
        case "mean":
            area = size * size

            for i in range(height):
                for j in range(width):

                    # area kernel
                    region = padded[i:i+size, j:j+size]

                    # hitung mean
                    canvas[i, j] = np.sum(region) / area

        # FILTER MEDIAN
        case "median":
            for i in range(height):
                for j in range(width):

                    # area kernel
                    region = padded[i:i+size, j:j+size]

                    # hitung median
                    canvas[i, j] = np.median(region)

        # FILTER MODUS / MODE FILTER
        case "modus":
            for i in range(height):
                for j in range(width):

                    # area kernel
                    region = padded[i:i+size, j:j+size]

                    # flatten array
                    values = region.ravel()

                    # hitung frekuensi
                    count = {}

                    for val in values:
                        if val in count:
                            count[val] += 1
                        else:
                            count[val] = 1

                    # cari nilai terbanyak
                    max_count = 0
                    mode_val = 0

                    for val, freq in count.items():
                        if freq > max_count:
                            max_count = freq
                            mode_val = val

                    # simpan hasil
                    canvas[i, j] = mode_val

    # kembalikan gambar
    return canvas

def convolution(img, kernel):

    # ukuran kernel
    size = kernel.shape[0]

    # ukuran padding
    pad_size = size // 2

    # tambah padding nol
    padded = np.pad(img, pad_size, mode='constant')

    # canvas hasil
    canvas = np.zeros_like(img).astype(np.float32)

    # dimensi gambar
    height, width = img.shape

    # loop baris
    for i in range(height):

        # loop kolom
        for j in range(width):

            # area kernel
            region = padded[i:i+size, j:j+size]

            # hitung konvolusi
            canvas[i, j] = np.sum(region * kernel)

    # kembalikan gambar
    return canvas

def edge_detection(img, kernelX, kernelY):

    # konvolusi arah x
    gx = convolution(img, kernelX)

    # konvolusi arah y
    gy = convolution(img, kernelY)

    # gabungkan gradient
    edge = np.abs(gx) + np.abs(gy)

    # normalisasi ke 0-255
    edge = (edge / edge.max()) * 255

    # ubah ke uint8
    edge = edge.astype(np.uint8)

    return edge

def dilasi(image, kernel):
    height, width = image.shape
    k_height, k_width = kernel.shape
    center = k_height//2
    hasil = np.zeros((height, width))

    for i in range(center, height-center):
        for j in range(center, width-center):
            if image[i,j] == 255:
                for k in range(k_height):
                    for l in range(k_width):
                        if kernel[k,l] == 1:
                            hasil[i+k-center,j+l-center] =255
            else:
                if hasil[i,j] !=255:
                    hasil[i,j] = 0 

    return hasil

def thickening(img, kernel, iterasi=1):
    hasil = img.copy()

    for _ in range(iterasi):
        hasil = dilasi(hasil, kernel)

    return hasil

def ekualisasi(citra):

    height, width = citra.shape

    # Histogram
    hist = np.zeros(256, dtype=int)

    # Hitung histogram citra
    for i in range(height):
        for j in range(width):
            hist[int(citra[i, j])] += 1

    # CDF
    cdf = np.zeros(256, dtype=int)
    cdf[0] = hist[0]

    # Hitung CDF
    for i in range(1, 256):
        cdf[i] = cdf[i - 1] + hist[i]

    # Normalisasi CDF
    cdf_normal = np.round(cdf * 255 / (height * width)).astype(np.uint8)

    # Hasil ekualisasi
    hasil = np.zeros_like(citra, dtype=np.uint8)

    # Terapkan hasil CDF normalisasi
    for i in range(height):
        for j in range(width):
            hasil[i, j] = cdf_normal[int (citra[i, j])]

    return hasil


## Membaca Dataset
Bagian ini mendeteksi lokasi root proyek dengan memeriksa keberadaan folder `dataset`. Setelah folder ditemukan, program membaca setiap subfolder sebagai label kelas citra. File yang diproses dibatasi pada ekstensi gambar seperti JPG, JPEG, PNG, dan BMP agar file selain gambar tidak ikut terbaca. Setiap citra yang berhasil dibaca disimpan ke dalam array `data`, sedangkan nama kelas dan nama file disimpan ke `labels` dan `file_name`. Data citra dibuat dengan `dtype=object` karena ukuran citra asli dapat berbeda-beda sebelum proses resize dilakukan.


In [9]:
# Deteksi root project
current_path = Path.cwd()

if (current_path / "dataset").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "dataset").exists():
    PROJECT_ROOT = current_path.parent
else:
    raise FileNotFoundError("Folder dataset tidak ditemukan.")

DATASET_DIR = PROJECT_ROOT / "dataset"

data = []
labels = []
file_name = []

valid_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

for sub_folder in os.listdir(DATASET_DIR):
    sub_folder_path = DATASET_DIR / sub_folder

    if not sub_folder_path.is_dir():
        continue

    for filename in os.listdir(sub_folder_path):
        img_path = sub_folder_path / filename

        if img_path.suffix.lower() not in valid_extensions:
            continue

        img = cv.imread(str(img_path))

        if img is None:
            print(f"Gagal membaca gambar: {img_path}")
            continue

        img = img.astype(np.uint8)

        data.append(img)
        labels.append(sub_folder)
        file_name.append(filename)

# Pakai dtype object karena ukuran gambar asli bisa beda-beda
data = np.array(data, dtype=object)
labels = np.array(labels)
file_name = np.array(file_name)

print("Jumlah data:", len(data))
print("Label:", np.unique(labels))

Jumlah data: 140
Label: ['catterpillar' 'snail']


## Fungsi Preprocessing
Bagian ini mendefinisikan beberapa variasi preprocessing yang akan dibandingkan hasilnya. Fungsi `resize_grayscale()` menyeragamkan ukuran citra menjadi 384 x 256 piksel dan mengubah citra berwarna menjadi grayscale. `prepo1` hanya melakukan resize dan grayscale sebagai baseline, sedangkan `prepo2` menambahkan median filter untuk mengurangi noise. `prepo3` menambahkan ekualisasi histogram setelah median filter untuk meningkatkan kontras. `prepo4` menggunakan Sobel untuk mendeteksi tepi, dan `prepo5` melanjutkan hasil Sobel dengan thresholding agar citra menjadi biner. Seluruh fungsi disimpan dalam `PREPROCESSING_METHODS` supaya dapat dipanggil otomatis dengan nama metode yang jelas.


In [10]:
TARGET_SIZE = (384, 256)

def resize_grayscale(image, target_size=TARGET_SIZE):
    resized = cv.resize(image, target_size)

    if len(resized.shape) == 3:
        gray = cv.cvtColor(resized, cv.COLOR_BGR2GRAY)
    else:
        gray = resized

    return gray.astype(np.uint8)


def prepo1(image):
    gray = resize_grayscale(image)
    return gray


def prepo2(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    return median


def prepo3(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    equ = ekualisasi(median)
    return equ


def prepo4(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    sobel = edge_detection(median, sobelX, sobelY)
    return sobel

def prepo5(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    sobel = edge_detection(median, sobelX, sobelY)
    threshold = thresholding(sobel, 33)
    return threshold

PREPROCESSING_METHODS = {
    "prepo1_resize+grayscale": prepo1,
    "prepo2_resize+grayscale+median": prepo2,
    "prepo3_resize+grayscale+median+equ": prepo3,
    "prepo4_resize+grayscale+median+sobel": prepo4,
    "prepo5_resize+grayscale+median+sobel+thresholding": prepo5
}

## Menyimpan Hasil Preprocessing
Bagian ini menjalankan seluruh metode preprocessing pada setiap citra yang telah dibaca dari dataset. Program mengambil fungsi dari `PREPROCESSING_METHODS`, memproses citra satu per satu, lalu memastikan nilai piksel tetap berada pada rentang 0 sampai 255. Hasil citra disimpan ke dalam folder `preprocessing_output` dengan struktur berdasarkan nama metode preprocessing dan label kelas. Struktur folder ini menjaga hasil setiap percobaan tetap terpisah sehingga mudah digunakan pada tahap ekstraksi fitur. Program juga menampilkan pesan proses dan peringatan apabila ada citra yang gagal disimpan.


In [11]:
OUTPUT_DIR = PROJECT_ROOT / "preprocessing_output"

for prepo_name, prepo_function in PREPROCESSING_METHODS.items():
    print(f"\nMemproses: {prepo_name}")

    for i in range(len(data)):
        image = data[i]
        label = labels[i]
        filename = file_name[i]

        processed_image = prepo_function(image)
        processed_image = np.clip(processed_image, 0, 255).astype(np.uint8)

        save_dir = OUTPUT_DIR / prepo_name / label
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / filename

        success = cv.imwrite(str(save_path), processed_image)

        if not success:
            print("Gagal simpan:", save_path)

    print(f"Selesai: {prepo_name}")

print("\nSemua preprocessing selesai.")
print("Hasil disimpan di:", OUTPUT_DIR.resolve())


Memproses: prepo1_resize+grayscale
Selesai: prepo1_resize+grayscale

Memproses: prepo2_resize+grayscale+median
Selesai: prepo2_resize+grayscale+median

Memproses: prepo3_resize+grayscale+median+equ
Selesai: prepo3_resize+grayscale+median+equ

Memproses: prepo4_resize+grayscale+median+sobel
Selesai: prepo4_resize+grayscale+median+sobel

Memproses: prepo5_resize+grayscale+median+sobel+thresholding
Selesai: prepo5_resize+grayscale+median+sobel+thresholding

Semua preprocessing selesai.
Hasil disimpan di: D:\Project-PCD-Kelompok-16\preprocessing_output


## Validasi Hasil Preprocessing
Bagian ini memeriksa kembali folder `preprocessing_output` setelah proses penyimpanan selesai. Program menelusuri setiap folder metode preprocessing dan setiap folder kelas di dalamnya. File yang dihitung hanya file dengan ekstensi gambar yang valid agar hasil validasi tidak tercampur file lain. Jumlah gambar pada setiap kelas ditampilkan untuk memastikan semua data telah diproses dan tersimpan. Validasi ini membantu menemukan kemungkinan folder kosong, data yang tidak lengkap, atau kesalahan saat proses penyimpanan.


In [12]:
valid_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

for prepo_dir in OUTPUT_DIR.iterdir():
    if not prepo_dir.is_dir():
        continue

    print(f"\n{prepo_dir.name}")

    for class_dir in prepo_dir.iterdir():
        if not class_dir.is_dir():
            continue

        image_files = [
            file for file in class_dir.iterdir()
            if file.suffix.lower() in valid_extensions
        ]

        print(f"- {class_dir.name}: {len(image_files)} gambar")


prepo1_resize+grayscale
- catterpillar: 70 gambar
- snail: 70 gambar

prepo2_resize+grayscale+median
- catterpillar: 70 gambar
- snail: 70 gambar

prepo3_resize+grayscale+median+equ
- catterpillar: 70 gambar
- snail: 70 gambar

prepo4_resize+grayscale+median+sobel
- catterpillar: 70 gambar
- snail: 70 gambar

prepo5_resize+grayscale+median+sobel+thresholding
- catterpillar: 70 gambar
- snail: 70 gambar
